In [42]:
import nltk
from spacy.tokenizer import Tokenizer
from spacy.lang.en import English
import spacy
from spacy.attrs import ORTH
from transformers import BertTokenizer
from collections import Counter
from matplotlib import pyplot as plt
import pandas as pd

### 1

In [2]:
def load_file_content():
    f = open("wsj_untokenized.txt", "r")
    content = f.readlines()
    # Preprocessing - text normalization
    # 1. Lowercase
    # 2. Lemmatization
    return content[0].lower()

In [3]:
def ntlk_tokenize(file_content):
    tokens = nltk.word_tokenize(file_content)
    types = set(tokens)
    ttr = len(set(tokens))/len(tokens)
    fdist = nltk.FreqDist(tokens)
    hapax_legomena = fdist.hapaxes()
    hapax_dislegomena = list(fdist.values()).count(2)
    return tokens, types, ttr, hapax_legomena, hapax_dislegomena

In [4]:
def spacy_tokenize(file_content):
    nlp = English()
    tokenizer = nlp.tokenizer
    tokens = tokenizer(file_content)
    types = { token.orth for token in tokens }
    ttr = len(set(types))/len(tokens)
    count_dict = tokens.count_by(ORTH)
    hapax_legomena = list(count_dict.values()).count(1)
    hapax_dislegomena = list(count_dict.values()).count(2)

    return tokens, types, ttr, hapax_legomena, hapax_dislegomena

In [5]:
def bert_tokenize(file_content):
    tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
    tokens = tokenizer.tokenize(file_content,  max_length=200000)
    types = set(tokens)
    ttr = len(set(tokens))/len(tokens)
    word_counts = Counter(tokens)
    hapax_legomena = list(word_counts.values()).count(1)
    hapax_dislegomena = list(word_counts.values()).count(2)

    return tokens, types, ttr, hapax_legomena, hapax_dislegomena

In [6]:
file_content = load_file_content()
nltk_tokens, nltk_types, nltk_ttr, nltk_hapax_legomena, nltk_hapax_dislegomena = ntlk_tokenize(file_content)
print("NLTK")
print(f"#Tokens: {len(nltk_tokens)}")
print(f"#Types: {len(set(nltk_tokens))}")
print(f"TTR: {len(set(nltk_tokens))/len(nltk_tokens)}")
print(f"Hapax legomena: {len(nltk_hapax_legomena)}")
print(f"Hapax dislegomena: {nltk_hapax_dislegomena}")

NLTK
#Tokens: 93356
#Types: 11017
TTR: 0.1180106259908308
Hapax legomena: 5615
Hapax dislegomena: 1702


In [7]:
spacy_tokens, spacy_types, spacy_ttr, spacy_hapax_legomena, spacy_hapax_dislegomena = spacy_tokenize(file_content)
print("spaCy")
print(f"#Tokens: {len(spacy_tokens)}")
print(f"#Types: {len(spacy_types)}")
print(f"TTR: {len(set(spacy_types))/len(spacy_tokens)}")
print(f"Hapax legomena: {spacy_hapax_legomena}")
print(f"Hapax dislegomena: {spacy_hapax_dislegomena}")

spaCy
#Tokens: 97152
#Types: 10445
TTR: 0.10751194005270093
Hapax legomena: 5072
Hapax dislegomena: 1659


In [8]:
bert_tokens, bert_types, bert_ttr, bert_hapax_legomena, bert_hapax_dislegomena = bert_tokenize(file_content)
print("Bert")
print(f"#Tokens: {len(bert_tokens)}")
print(f"#Types: {len(bert_types)}")
print(f"TTR: {bert_ttr}")
print(f"Hapax legomena: {bert_hapax_legomena}")
print(f"Hapax dislegomena: {bert_hapax_dislegomena}")
print("============")

Bert
#Tokens: 117968
#Types: 8544
TTR: 0.07242642072426421
Hapax legomena: 2915
Hapax dislegomena: 1381


---

### 2

In [9]:
def tokenize_nltk(sentence):
    nltk_tokens = nltk.word_tokenize(sentence)
    print(f"NLTK tokens:\n{nltk_tokens}\n===============")

In [10]:
def tokenize_spacy(sentence):
    nlp = English()
    tokenizer = nlp.tokenizer
    spacy_tokens = tokenizer(sentence)
    st = [ str(token) for token in spacy_tokens ]
    print(f"spaCy tokens:\n{st}\n===============")

In [11]:
def tokenize_bert(sentence):
    tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
    bert_tokens = tokenizer.tokenize(sentence,  max_length=200000)
    print(f"BERT tokens:\n{bert_tokens}\n===============")

In [12]:
sentence = "Pierre Vinken, 61 years old, will join the board as a nonexecutive director Nov. 29."
tokenize_nltk(sentence)
tokenize_spacy(sentence)
tokenize_bert(sentence)

NLTK tokens:
['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.']
spaCy tokens:
['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.']
BERT tokens:
['Pierre', 'Vin', '##ken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'none', '##xe', '##cut', '##ive', 'director', 'Nov', '.', '29', '.']


---

### 3

#### NLTK

In [15]:
nltk_appearance_threshold = int(0.3*len(nltk_tokens))
nltk_fdist = nltk.FreqDist(nltk_tokens)
nltk_token_collection = {}
nltk_total_appearance_count = 0
for tkn, freq in nltk_fdist.most_common():
    if nltk_total_appearance_count + freq < nltk_appearance_threshold:
        nltk_token_collection[tkn] = freq
        nltk_total_appearance_count += freq
    else:
        break

In [16]:
print(f"NLTK\n30% of token count:{nltk_appearance_threshold}\nType apperance sum: {nltk_total_appearance_count}.")

NLTK
30% of token count:28006
Type apperance sum: 27882.


#### spaCY

In [17]:
spacy_appearance_threshold = int(0.3*len(spacy_tokens))
# Convert to list of strings for FreqDist() to work
spacy_token_strings = [ token.text for token in spacy_tokens ]
spacy_fdist = nltk.FreqDist(spacy_token_strings)
spacy_token_collection = {}
spacy_total_appearance_count = 0
for tkn, freq in spacy_fdist.most_common():
    if spacy_total_appearance_count + freq < spacy_appearance_threshold:
        spacy_token_collection[tkn] = freq
        spacy_total_appearance_count += freq
    else:
        break

In [18]:
print(f"spaCy\n30% of token count:{spacy_appearance_threshold}\nType apperance sum: {spacy_total_appearance_count}.")

spaCy
30% of token count:29145
Type apperance sum: 28729.


#### BERT

In [19]:
bert_appearance_threshold = int(0.3*len(bert_tokens))
bert_fdist = nltk.FreqDist(bert_tokens)
bert_token_collection = {}
bert_total_appearance_count = 0
for tkn, freq in bert_fdist.most_common():
    if bert_total_appearance_count + freq < bert_appearance_threshold:
        bert_token_collection[tkn] = freq
        bert_total_appearance_count += freq
    else:
        break

In [20]:
print(f"BERT\n30% of token count:{bert_appearance_threshold}\nType apperance sum: {bert_total_appearance_count}.")

BERT
30% of token count:35390
Type apperance sum: 35265.


In [30]:
# Token count
print(list(map(len, [nltk_token_collection, spacy_token_collection, bert_token_collection])))

[14, 12, 13]


In [110]:
# Plotting
data = {
    "NLTK Type": list(nltk_token_collection.keys()),
    "NLTK Freq": list(nltk_token_collection.values()),
    "spaCy Type": list(spacy_token_collection.keys()),
    "spaCy Freq": list(spacy_token_collection.values()),
    "BERT Type": list(bert_token_collection.keys()),
    "BERT Freq": list(bert_token_collection.values())
}

df = pd.DataFrame(dict([(key, pd.Series(value)) for key, value in data.items()]))
df.style.set_properties(**{"text-align": "center"})

# Fix column types to avoid trailing zeroes
freq_columns = ["NLTK Freq", "spaCy Freq", "BERT Freq"]
for col in freq_columns:
    df[col] = df[col].astype("Int32")

In [111]:
df

,NLTK Type,NLTK Freq,spaCy Type,spaCy Freq,BERT Type,BERT Freq
0,",",4823,.,5020,.,6363
1,the,4757,",",4823,",",5026
2,.,3645,the,4763,the,4767
3,of,2318,of,2319,',4117
4,to,2175,to,2180,of,2320
5,a,1967,a,1991,to,2226
6,in,1760,in,1773,a,2150
7,and,1534,and,1541,in,1940
8,'',959,'',1372,-,1733
9,'s,865,-,1231,and,1549


In [129]:
# Find type intersection
nltk_set = set(nltk_token_collection.keys())
spacy_set = set(spacy_token_collection.keys())
bert_set = set(bert_token_collection.keys())

intersection = nltk_set.intersection(spacy_set, bert_set)
print(f"Common types in all methods: {len(intersection)}")

Common types in all methods: 9


---